### Installing packages

In [2]:
# we use natsort package to sort those missing leading zero files 
# !pip install natsort

### Defining ANSI codes for colored text prints 

In [1]:
# ANSI escape codes, to make log prints nicer
RED = "\033[31m"
GREEN = "\033[32m"
YELLOW = "\033[33m"
BLUE = "\033[34m"
BOLD = "\033[1m"
ITALIC = "\x1B[3m"
UNDERLINED = "\033[4m"
RESET = "\033[0m"
WHITE_BG    = "\x1b[47m\033[30m" # adding \033[30m makes text black
GREEN_BG    = "\x1b[102m\033[30m" # adding \033[30m makes text black

# https://jakob-bagterp.github.io/colorist-for-python/ansi-escape-codes/standard-16-colors/#bright-colors_2

### Imports

In [2]:
import pandas as pd
import os
from natsort import os_sorted
from datetime import datetime, timedelta
import requests
import zipfile
import warnings 
import urllib3

from dotenv import dotenv_values
from sqlalchemy import create_engine, types, text

# we'll suppress the "missing SSL certificate" warnings while downloading files
warnings.simplefilter("ignore", urllib3.exceptions.InsecureRequestWarning) 

## Data Download
**Sources**  
>Raw Data: https://transtats.bts.gov/PREZIP/  
>Website: https://transtats.bts.gov

#### working scenario: 
1. choose a time period for your flights data<br>**NOTE:** usually latest month available is = now - 3 months
2. in the first cell: 
    - update `start` for the start date
    - update `length` for the number of month 
3. execute all other cells in this notebook
   <br>**NOTE:** the steps are optimized for multiple months period, but would also work for 1 month  
  
<details>
<summary style="color:grey">all steps explained</summary>

1. decide on the period and update `start` and `length` variables
2. if not yet created, add 2 folders inside `\da-analytics-engineering-project\` repo:
     - `downloads`
     - and `downloads/extracted`
3. choose the time period for the flights data (starting month, total number of months)    
4. under the [transtats URL](https://transtats.bts.gov/PREZIP/) above find files names starting with  
`"On_Time_Reporting_Carrier_On_Time_Performance_1987_present_####_##.zip"`  
- each ZIP file contains a CSV file for **one month** of data (indicated as ####_##)  
- download desired zipfiles to the `downloads` folder  
5. extract the CSV files into the `downloads/extracted` folder
</details>

In [3]:
# 1. Decide on starting month and total number of months
# additional data to compare this later to 2023 (our target month: February 2023)
start = '02.2023' # Enter the starting month and the year (MM.YYYY)
length = 1 # How many months do you need?

In [4]:
# 2. Create folders for the zip files download and for the CSV-files extraction
os.makedirs('./downloads/extracted', exist_ok=True)

In [5]:
# 3. Create a list of months for the flight

# Generate list of MM.YYYY values for one year
def generate_year_list(start, length):
    start_date = datetime.strptime(start, '%m.%Y')
    return [f"{dt.year}_{dt.month}" for dt in
        (start_date + timedelta(days=31 * i) for i in range(length))]

# MM_YYYY values for the period lenght
year_month_list = generate_year_list(start, length)

print(year_month_list)


['2023_2']


In [15]:
# 4. Download ZIP files (~35 seconds per one file)

# Define the URL of the ZIP file
base_url = 'https://transtats.bts.gov/PREZIP/'
download_time = timedelta(0) # for time logging
disk_space_zip = 0

for year_month in year_month_list:

    # Define the URL of the ZIP file and the CSV file
    zip_name = f'On_Time_Reporting_Carrier_On_Time_Performance_1987_present_{year_month}.zip'

    print(f'\n ⏳ This should take {RED}~35 seconds...{RESET}\n\n    ⬇️ {BLUE}downloading:{RESET} {zip_name}')
    print(f'    🐌 {YELLOW}wait for it...{RESET}', end='\r')
    start_time = datetime.now()

    # Send a HTTP request to the specified URL and save the response content
    response = requests.get(base_url+zip_name, verify=False) # we ignore the SSL certificate warnings

    with open(f'./downloads/{zip_name}', 'wb') as file: # save the ZIP in "downloads folder"
        file.write(response.content)
        print(f'    ✅ {GREEN}file saved:{RESET} {zip_name}', end=' ')
    
    # assessing the size of the downloaded file
    file_size = os.path.getsize(f'./downloads/{zip_name}') 
    size_in_mb = file_size / (1024 ** 2) 
    print(f'{GREEN}({size_in_mb:.2f} MB){RESET}\n')
    disk_space_zip += file_size

    # just some fun with basic time logging  
    end_time = datetime.now()
    time_difference = end_time - start_time
    download_time = download_time + time_difference
    if (time_difference.seconds // 60) < 1:
        print(f' 🦊 Actually it took: {YELLOW}{time_difference.seconds % 60} seconds\n{RESET}','-'*80)
    else:
        print(f' 🦊 Actually it took: {YELLOW}{time_difference.seconds // 60} minutes and {time_difference.seconds % 60} seconds\n{RESET}','-'*80)
print(f' 🦊 Total Download Time: {YELLOW}{download_time.seconds // 60} minutes and {download_time.seconds % 60} seconds\n{RESET}')
print(f' 🐹 Used Disk Space: {GREEN}({(disk_space_zip / (1024 ** 2)):.2f} MB){RESET}')


 ⏳ This should take ~35 seconds...

    ⬇️ downloading: On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2023_2.zip
    ✅ file saved: On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2023_2.zip (23.59 MB)

 🦊 Actually it took: 32 seconds
 --------------------------------------------------------------------------------
 🦊 Total Download Time: 0 minutes and 32 seconds

 🐹 Used Disk Space: (23.59 MB)


In [16]:
# 5. Extracting CSV files only

disk_space_csv = 0

for year_month in year_month_list:

    # Define the name of the ZIP file and the CSV file
    zip_name = f'On_Time_Reporting_Carrier_On_Time_Performance_1987_present_{year_month}.zip'
    csv_name = f'On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_{year_month}.csv'

    # Open the downloaded ZIP file
    with zipfile.ZipFile(f'./downloads/{zip_name}', 'r') as zip_ref:
        # Extract the CSV file
        zip_ref.extract(csv_name, path='./downloads/extracted/') # save the CSV in "downloads folder"
        print(f'    🍌 extracted "{csv_name}', end=' ')
        
    # assessing the size of the extracted file
    file_size = os.path.getsize(f'./downloads/extracted/{csv_name}') 
    size_in_mb = file_size / (1024 ** 2) 
    print(f"{GREEN}({size_in_mb:.2f} MB){RESET}\n")
    disk_space_csv += file_size

print('-'*80,f'\n 🐹 Used Disk Space: {GREEN}({(disk_space_csv / (1024 ** 2)):.2f} MB){RESET}')


    🍌 extracted "On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2023_2.csv (216.21 MB)

-------------------------------------------------------------------------------- 
 🐹 Used Disk Space: (216.21 MB)


# Data Wrangling

### 1. adding all CSV file names to a list

In [17]:
# Add all file names from the "extracted" folder to a list
file_names = os.listdir('./downloads/extracted/')

# make sure only the data files are in the list
file_names_unordered = [fname for fname in file_names if fname.startswith("On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_")]

# using os_sorted function (from natsort) - able to sort strings with numbers ['2','1','11']
# sorted(['2','1','11']) # for comparison
data_files = os_sorted(file_names_unordered)

data_files

['On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2022_2.csv',
 'On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2023_2.csv']

In [18]:
# we have 110 columns in each CSV...
file_check = pd.read_csv(f'./downloads/extracted/{data_files[0]}', low_memory = False)
file_check.shape

(495713, 110)

In [19]:
# original column names are not optimal and need renaming...
print(file_check.columns.to_list())

['Year', 'Quarter', 'Month', 'DayofMonth', 'DayOfWeek', 'FlightDate', 'Reporting_Airline', 'DOT_ID_Reporting_Airline', 'IATA_CODE_Reporting_Airline', 'Tail_Number', 'Flight_Number_Reporting_Airline', 'OriginAirportID', 'OriginAirportSeqID', 'OriginCityMarketID', 'Origin', 'OriginCityName', 'OriginState', 'OriginStateFips', 'OriginStateName', 'OriginWac', 'DestAirportID', 'DestAirportSeqID', 'DestCityMarketID', 'Dest', 'DestCityName', 'DestState', 'DestStateFips', 'DestStateName', 'DestWac', 'CRSDepTime', 'DepTime', 'DepDelay', 'DepDelayMinutes', 'DepDel15', 'DepartureDelayGroups', 'DepTimeBlk', 'TaxiOut', 'WheelsOff', 'WheelsOn', 'TaxiIn', 'CRSArrTime', 'ArrTime', 'ArrDelay', 'ArrDelayMinutes', 'ArrDel15', 'ArrivalDelayGroups', 'ArrTimeBlk', 'Cancelled', 'CancellationCode', 'Diverted', 'CRSElapsedTime', 'ActualElapsedTime', 'AirTime', 'Flights', 'Distance', 'DistanceGroup', 'CarrierDelay', 'WeatherDelay', 'NASDelay', 'SecurityDelay', 'LateAircraftDelay', 'FirstDepTime', 'TotalAddGTime'

### 2. Defining functions
<font size=4>
<ul><li>column filter<li>renaming columns<li>changing data types</ul>
</font>

In [20]:
# select columns to keep
def cols_to_keep(flights_raw):
    columns_to_keep = [
        "Year",
        "Month",
        "FlightDate",
        "DepTime",
        "CRSDepTime",
        "DepDelay",
        "ArrTime",
        "CRSArrTime",
        "ArrDelay",
        "Reporting_Airline",
        "Tail_Number",
        "Flight_Number_Reporting_Airline",
        "Origin",
        "Dest",
        "AirTime",
        "ActualElapsedTime",
        "Distance",
        "Cancelled",
        "Diverted",
    ]
    flights = flights_raw.loc[:, columns_to_keep]
    return flights

In [21]:
# rename columns
def rename_cols(flights):
    new_column_names = {
        'Year': 'year',
        'Month': 'month',
        'FlightDate': 'flight_date',
        'DepTime': 'dep_time',
        'CRSDepTime': 'sched_dep_time',
        'DepDelay': 'dep_delay',
        'ArrTime': 'arr_time',
        'CRSArrTime': 'sched_arr_time',
        'ArrDelay': 'arr_delay',
        'Reporting_Airline': 'airline',
        'Tail_Number': 'tail_number',
        'Flight_Number_Reporting_Airline': 'flight_number',
        'Origin': 'origin',
        'Dest': 'dest',
        'AirTime': 'air_time',
        'ActualElapsedTime': 'actual_elapsed_time',
        'Distance': 'distance',
        'Cancelled': 'cancelled',
        'Diverted': 'diverted'
    }
    flights.rename(columns=new_column_names, inplace=True)
    return flights

In [22]:
# change datatype
def change_dtypes(flights):
    types_change = {
        'flight_date': 'datetime64[ns]',
        'dep_time': 'Int16',
        'sched_dep_time': 'Int16',
        'dep_delay': 'Int16',
        'arr_time': 'Int16',
        'sched_arr_time': 'Int16',
        'arr_delay': 'Int16',
        'airline': 'O',
        'tail_number': 'O',
        'flight_number': 'Int16',
        'origin': 'O',
        'dest': 'O',
        'air_time': 'Int16',
        'actual_elapsed_time': 'Int16',
        'distance': 'Int16',
        'cancelled': 'Int16',
        'diverted': 'Int16'
    }
    flights = flights.astype(types_change)
    return flights

### 3. for-loop over the `data_files` list:
<font size=4>
<ol>
<li>reading a CSV as dataframe
<li>filtering columns
<li>renaming columns
<li>changing data types
<li>append dataframe to a list of dataframes
</ol>
</font>

In [23]:
data_files

['On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2022_2.csv',
 'On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2023_2.csv']

In [24]:
# list for separate dataframes
flights_list = []

#  loop over the extracted csv files and execute functions 
for file in data_files:
    print(file)
    # 1. read as a dataframe
    print('reading...', end=" ")
    flights_raw = pd.read_csv(f'./downloads/extracted/{file}', low_memory = False) 

    # 2.select columns to keep
    flights_select = cols_to_keep(flights_raw) 
    print('filter colums...', end=" ")
    
    # 3. rename columns
    flights_rename = rename_cols(flights_select) 
    print('rename colums...', end=" ")

    # 4. change data types
    flights_dtypes = change_dtypes(flights_rename) 
    print('change dtypes...', end=" ")
    
    # 5. add to the list of dateframes
    flights_list.append(flights_dtypes) 
    print(f'✅ {GREEN}appended to flight_list{RESET}\n')
    
print(f'Done. The list has {len(flights_list)} elements')

On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2022_2.csv
reading... filter colums... rename colums... change dtypes... ✅ appended to flight_list

On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2023_2.csv
reading... filter colums... rename colums... change dtypes... ✅ appended to flight_list

Done. The list has 2 elements


In [25]:
# 6. concatenate the list of dataframes to a one dataframe
flights_all = pd.concat(flights_list)

In [26]:
# sort dataframe 
flights_all.sort_values(['flight_date','sched_dep_time'], inplace=True)
flights_all

,year,month,flight_date,dep_time,sched_dep_time,dep_delay,arr_time,sched_arr_time,arr_delay,airline,tail_number,flight_number,origin,dest,air_time,actual_elapsed_time,distance,cancelled,diverted
24938,2022,2,2022-02-01,2400,2,-2,442,501,-19,AA,N587UW,496,LAX,DFW,139,162,1235,0,0
362027,2022,2,2022-02-01,2349,10,-21,400,437,-37,UA,N69847,1947,PHX,ORD,164,191,1440,0,0
211411,2022,2,2022-02-01,6,15,-9,602,621,-19,NK,N680NK,1358,LAX,ORD,199,236,1744,0,0
183114,2022,2,2022-02-01,39,20,19,610,559,11,F9,N207FR,486,DEN,RSW,185,211,1607,0,0
183224,2022,2,2022-02-01,16,21,-5,712,711,1,F9,N339FR,2000,LAS,ATL,196,236,1747,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
449439,2023,2,2023-02-28,2350,2359,-9,538,553,-15,UA,N17753,2497,SEA,ORD,194,228,1721,0,0
449513,2023,2,2023-02-28,1,2359,2,528,523,5,UA,N78540,2423,SFO,AUS,170,207,1504,0,0
449555,2023,2,2023-02-28,2343,2359,-16,544,614,-30,UA,N69826,2379,SEA,IAH,221,241,1874,0,0
450693,2023,2,2023-02-28,2352,2359,-7,337,423,-46,UA,N69840,1126,PHX,ORD,144,165,1440,0,0


In [27]:
# countercheck the time period
flights_all['flight_date'].min(), flights_all['flight_date'].max()

(Timestamp('2022-02-01 00:00:00'), Timestamp('2023-02-28 00:00:00'))

# Saving the combined dataset<p><font size=5>(just as a backup)</font>

In [28]:
# define the file name for the combined CSV file (using period's first and last month)
output_file_name = f'flights_from_{year_month_list[0]}_until_{year_month_list[-1]}.csv'
output_file_name

'flights_from_2023_2_until_2023_2.csv'

In [29]:
# create folder 'data'
os.makedirs('./data', exist_ok=True)

In [30]:
# saving
flights_all.to_csv(f'./data/{output_file_name}', index=False)

print(f' ✅ {GREEN}Combined Dataset Saved:{RESET} {output_file_name}', end=' ')

# assessing the size of the extracted file
file_size = os.path.getsize(f'./data/{output_file_name}') 
size_in_mb = file_size / (1024 ** 2) 
print(f"{GREEN}({size_in_mb:.2f} MB){RESET}\n")

# Get the absolute path
absolute_path = os.path.abspath(f'./data/{output_file_name}')
print(absolute_path)


 ✅ Combined Dataset Saved: flights_from_2023_2_until_2023_2.csv (76.97 MB)

/Users/dariaskibo/Spiced_Files/da-cayenne-clusters-m5-analytics-engineering-project/data/flights_from_2023_2_until_2023_2.csv


### We got the data in a dataframe. Now it needs to be loaded into our DB.

_________

### Next Steps: 
1. Reduce your dataframe `flights_all` to 3-5 origin/dest airports affected by the weather event (check if they have weather stations here: https://meteostat.net/en/). You can expand your dataset to include more locations or destinations if this is helpful in your analysis. But keep an eye on the size of your data. Dealing with GBs of raw data can make everything very slow.
2. Using the Lecture `03_sql_with_python.ipynb` as an example 
   - load DB credentials from your `.env` file
   - define a connection string
   - create an engine (SQLAlchemy)
   - set the **search_path** to your project schema
   - define data types as `flights_dtypes` (<font style="color:lime">prepared for you here below</font>)
   - upload filtered `flights_all` dataframe to your project schema in our database <br>(<font style="color:lime">pass the `dtype=flights_dtypes` argument</font>)

3. You can create a copy of the table `airports` from the schema `public`<p>HINT: 
   - You could run a query via your SQL Alchemy engine 
   - or to be quick, you can run a query in DBeaver


In [ ]:
from sqlalchemy import create_engine, types
from sqlalchemy import text

In [33]:
from dotenv import dotenv_values

config = dotenv_values()

# define variables for the login
pg_user = config['POSTGRES_USER']  # align the key label with your .env file !
pg_host = config['POSTGRES_HOST']
pg_port = config['POSTGRES_PORT']
pg_db = config['POSTGRES_DB']
pg_schema = config['POSTGRES_SCHEMA']
pg_pass = config['POSTGRES_PASS']

In [34]:
# build the link:
url = f'postgresql://{pg_user}:{pg_pass}@{pg_host}:{pg_port}/{pg_db}'

# create an instance of an engine with my schema:
engine = create_engine(url, 
                       echo=True, 
                       connect_args={'options': f'-c search_path={pg_schema}'}) 
# this parameter takes in the schema and uses it every time when the engine runs

In [35]:
pg_schema

'aero_insights'

In [36]:
# defining data types for the DB
flights_dtypes = {
    'flight_date': types.DateTime,
    'dep_time': types.Integer,
    'sched_dep_time': types.Integer,
    'dep_delay': types.Integer,
    'arr_time': types.Integer,
    'sched_arr_time': types.Integer,
    'arr_delay': types.Integer,
    'airline': types.String,
    'tail_number': types.String,
    'flight_number': types.Integer,
    'origin': types.String,
    'dest': types.String,
    'air_time': types.Integer,
    'actual_elapsed_time': types.Integer,
    'distance': types.Integer,
    'cancelled': types.Integer,
    'diverted': types.Integer
}

flights_all.to_sql(
    name='flights_all', 
    con=engine, 
    schema=schema,      # Explicitly state the schema
    if_exists='replace', # Or 'append'
    index=False,        # Usually best to avoid uploading the pandas index
    dtype=flights_dtypes # Using the prepared dtypes
)

In [41]:
flights_all.columns

Index(['year', 'month', 'flight_date', 'dep_time', 'sched_dep_time',
       'dep_delay', 'arr_time', 'sched_arr_time', 'arr_delay', 'airline',
       'tail_number', 'flight_number', 'origin', 'dest', 'air_time',
       'actual_elapsed_time', 'distance', 'cancelled', 'diverted'],
      dtype='str')

In [39]:
flights_all.dtypes

year                            int64
month                           int64
flight_date            datetime64[ns]
dep_time                        Int16
sched_dep_time                  Int16
dep_delay                       Int16
arr_time                        Int16
sched_arr_time                  Int16
arr_delay                       Int16
airline                        object
tail_number                    object
flight_number                   Int16
origin                         object
dest                           object
air_time                        Int16
actual_elapsed_time             Int16
distance                        Int16
cancelled                       Int16
diverted                        Int16
dtype: object

In [43]:
# filter the airports we need: JFK, BOS, ORD, EWR
airports_need = ['JFK','BOS', 'ORD', 'EWR']

flights_filtered = flights_all[
    (flights_all['origin'].isin(airports_need)) |
    (flights_all['dest'].isin(airports_need))
].copy()

In [46]:
flights_all.shape

(998462, 19)

In [45]:
flights_filtered.shape

(192114, 19)

In [48]:
len(flights_filtered['origin'].unique())

172

In [49]:
len(flights_filtered['dest'].unique())

173

In [50]:
# this it to write:
flights_filtered.to_sql(name='flights_all', 
                        con=engine, 
                        schema='aero_insights', 
                        if_exists='append',
                        index=False, 
                        dtype=flights_dtypes)

2026-04-21 13:04:37,327 INFO sqlalchemy.engine.Engine select pg_catalog.version()
2026-04-21 13:04:37,328 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-04-21 13:04:37,366 INFO sqlalchemy.engine.Engine select current_schema()
2026-04-21 13:04:37,367 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-04-21 13:04:37,398 INFO sqlalchemy.engine.Engine show standard_conforming_strings
2026-04-21 13:04:37,398 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-04-21 13:04:37,430 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-04-21 13:04:37,444 INFO sqlalchemy.engine.Engine SELECT pg_catalog.pg_class.relname 
FROM pg_catalog.pg_class JOIN pg_catalog.pg_namespace ON pg_catalog.pg_namespace.oid = pg_catalog.pg_class.relnamespace 
WHERE pg_catalog.pg_class.relname = %(table_name)s AND pg_catalog.pg_class.relkind = ANY (ARRAY[%(param_1)s, %(param_2)s, %(param_3)s, %(param_4)s, %(param_5)s]) AND pg_catalog.pg_namespace.nspname = %(nspname_1)s
2026-04-21 13:04:37,445 INFO sqlalchemy.engine.Engine [g

114